# 📚 Конспект: Модулі, Імпорти та CLI в Python

> Урок 5 — повний конспект з теорією, розширеними прикладами, задачами та тестом

---

## 📋 Зміст

| # | Тема | Ключові концепції |
|---|------|-------------------|
| 1 | Механізм імпорту | `import`, виконання коду, `sys.modules` |
| 2 | Захист `__name__` | `if __name__ == "__main__":` |
| 3 | Пошук модулів | `sys.path`, тіньовий імпорт |
| 4 | Аргументи CLI | `sys.argv`, типи, валідація |
| 5 | CLI-програми | `while True:`, `input()`, методи рядка |
| 6 | Шпаргалка | швидкий довідник |
| 7 | Задачі | TODO самостійна робота |
| 8 | Тест | автоматична перевірка |

In [ ]:
import sys
for path in sys.path:
    print("path---->", path)

In [ ]:
for path in sys.path:
    if "site-packages" in path:
        print("Libraries installed here:", path)

In [ ]:
sys.path.append("my_modules")

print(sys.path)

In [ ]:
print("Python version:", sys.version)
print("Executable:", sys.executable)
print("Module search paths:")

for path in sys.path:
    print("  ", path)

---

## 📦 Частина 1: Механізм імпорту

### Що відбувається при `import`?

Коли Python зустрічає `import something`, він виконує **три кроки**:

1. **Перевіряє кеш** `sys.modules` — якщо модуль вже завантажений, повертає його (не перечитує файл)
2. **Знаходить файл** `something.py` за списком шляхів у `sys.path`
3. **Читає файл зверху донизу** і **виконує весь код верхнього рівня**

```
┌────────────────────────────────────────────────────────────┐
│                    import greeter                          │
├──────────────────────────────┬─────────────────────────────┤
│  greeter.py                  │  Що відбувається?           │
│  ────────────────────────    │  ──────────────────────     │
│  print("Завантажено!")       │  ← ВИКОНУЄТЬСЯ одразу       │
│  GREETING = "Привіт"         │  ← ВИКОНУЄТЬСЯ одразу       │
│  def greet(name):            │  ← функція СТВОРЮЄТЬСЯ      │
│      return f"..."           │  ← тіло НЕ виконується      │
│  class Greeter: ...          │  ← клас СТВОРЮЄТЬСЯ         │
└──────────────────────────────┴─────────────────────────────┘
```

### Кешування модулів (`sys.modules`)

Python **не перечитує файл** при повторному `import` — модуль кешується:

```python
import greeter   # читає файл, виконує top-level код
import greeter   # використовує кеш — нічого не виконує
import greeter   # те саме
```

### Форми імпорту

| Синтаксис | Доступ до функції | Коли використовувати |
|-----------|-------------------|---------------------|
| `import math` | `math.sqrt(4)` | потрібен весь модуль |
| `from math import sqrt` | `sqrt(4)` | конкретні функції |
| `from math import sqrt, pi` | `sqrt(4)`, `pi` | кілька функцій |
| `from math import *` | `sqrt(4)` | ⚠️ уникайте — засмічує простір імен |
| `import numpy as np` | `np.array([1,2])` | довгі імена модулів |

### ⚠️ Небезпека: Побічні ефекти при імпорті

```python
# ❌ Погано — побічний ефект виконається при кожному першому import
print("Модуль завантажено!")   # виконається при import!
connect_to_database()          # теж виконається!

# ✅ Добре — тільки def / class / константи на верхньому рівні
DB_HOST = "localhost"

def connect():
    ...
```

**Правило:** На верхньому рівні модуля розміщуйте тільки `def`, `class`, константи та `import`.

In [ ]:
# Приклад 1.1: Що виконується при import
# Симулюємо вміст уявного модуля та аналізуємо, що відбувається

print("=== Аналіз: що виконується при import ===")
print()

# Уявімо, що це вміст файлу module_demo.py
module_code = '''
print("[TOP-LEVEL] Рядок 1 — виконується!")   # print на верхньому рівні

GREETING = "Привіт"                             # присвоєння — виконується

print("[TOP-LEVEL] Рядок 3 — теж виконується")

def greet(name):                                 # def — функція СТВОРЮЄТЬСЯ
    print("[INSIDE FUNC] Це НЕ виконується при import")
    return f"{GREETING}, {name}!"

class MyClass:                                   # class — СТВОРЮЄТЬСЯ
    def method(self):                            # тіло класу — НЕ виконується
        pass
'''

print("📄 Вміст module_demo.py:")
print(module_code)
print()
print("▶ При import виконуються:")
print("  ✅ print() на верхньому рівні")
print("  ✅ GREETING = 'Привіт'")
print("  ✅ def greet — функція створюється в пам'яті")
print("  ✅ class MyClass — клас створюється")
print("  ❌ тіло greet() — НЕ виконується (тільки при виклику)")
print("  ❌ тіло MyClass.method() — НЕ виконується")

In [ ]:
# Приклад 1.2: Кеш модулів sys.modules
import sys

print("=== Кеш модулів sys.modules ===")
print()

# Перевіряємо чи є math в кеші
print("1. До першого import math:")
print(f"   'math' in sys.modules: {'math' in sys.modules}")
print()

import math
print("2. Після import math:")
print(f"   'math' in sys.modules: {'math' in sys.modules}")
print(f"   sys.modules['math']: {sys.modules['math']}")
print()

# Повторний import НЕ перечитує файл
import math  # використовує кеш
print("3. Повторний 'import math' — використано кеш (файл не перечитується)")
print()

# Примусовий перезапуск через importlib
import importlib
importlib.reload(math)
print("4. Після importlib.reload(math) — файл перечитано")
print("   (корисно при розробці, коли ви змінили файл модуля)")

In [ ]:
# Приклад 1.3: Різні форми імпорту
import math                        # весь модуль
from math import sqrt, pi          # конкретні об'єкти
import math as m                   # псевдонім
from math import factorial as fact # псевдонім для функції

print("=== Різні форми імпорту ===")
print()
print(f"import math           → math.sqrt(16)   = {math.sqrt(16)}")
print(f"from math import sqrt → sqrt(16)        = {sqrt(16)}")
print(f"import math as m      → m.sqrt(16)      = {m.sqrt(16)}")
print(f"factorial as fact     → fact(5)         = {fact(5)}")
print()
print(f"Константа pi: {pi:.6f}")
print()
print("⚠️  Різниця між формами — тільки синтаксис доступу.")
print("    Модуль завантажується однаково в усіх випадках.")

---

## 🛡️ Частина 2: Захист через `__name__`

### Що таке `__name__`

У кожному Python-файлі є спеціальна змінна:

```python
__name__
```

Python **створює її автоматично**, щоб файл міг зрозуміти **як його запустили**.

Файл може працювати у **двох режимах**.

---

### 1️⃣ Прямий запуск файлу

Якщо запустити файл напряму:

```bash
python my_math.py
```

Python встановлює:

```python
__name__ = "__main__"
```

Це означає, що файл є **головною програмою**.

---

### 2️⃣ Імпорт файлу

Якщо інший файл імпортує цей код:

```python
import my_math
```

Python встановлює:

```python
__name__ = "my_math"
```

Тобто значення дорівнює **назві модуля**.

---

| Контекст | Значення `__name__` | Що відбувається |
|----------|--------------------|-----------------|
| `python my_math.py` (прямий запуск) | `"__main__"` | файл — це **точка входу** |
| `import my_math` | `"my_math"` | файл — це **модуль-бібліотека** |

## Патерн `if __name__ == "__main__":`

Це перевірка:

> **Чи запущений файл напряму?**

```python
if __name__ == "__main__":
    print("Файл запущено напряму")
```

Якщо файл **імпортували**, цей код **не виконується**.
`

```python
# my_math.py

def add(a, b):
    return a + b

def square(x):
    return x * x

if __name__ == "__main__":    # ← виконується ТІЛЬКИ при прямому запуску
    print("=== Тестування my_math ===")
    print("add(2, 3) =", add(2, 3))
    print("square(4) =", square(4))
```

### Навіщо це потрібно?

```
Файл my_math.py — подвійне призначення:

  ┌─────────────────────────────────────────────────────────────┐
  │  python my_math.py     →  __name__ = "__main__"             │
  │                           блок тестування ВИКОНУЄТЬСЯ       │
  │                                                             │
  │  import my_math         →  __name__ = "my_math"             │
  │                           блок тестування НЕ виконується    │
  └─────────────────────────────────────────────────────────────┘
```

**Переваги:**
- Файл є **і бібліотекою, і програмою** одночасно
- Тести запускаються тільки при прямому виклику
- Немає побічних ефектів при імпорті

In [ ]:
# Приклад 2.1: Перевірка __name__ у поточному контексті
print("=== Змінна __name__ ===")
print()
print(f"Значення __name__ у Jupyter: {__name__!r}")
print()
print("Висновок:")
if __name__ == "__main__":
    print("  Цей код запущений як основна програма (або в Jupyter)")
else:
    print("  Цей код імпортований як модуль")
print()

# Перевіримо __name__ у модулях
import math
import os
print(f"math.__name__  = {math.__name__!r}")
print(f"os.__name__    = {os.__name__!r}")
print()
print("Висновок: ім'я модуля = назва файлу без .py")

In [ ]:
# Приклад 2.2: Файл my_math.py — як бібліотека і як програма
import my_math

print("=== Використання my_math як бібліотеки ===")
print()
print(f"my_math.add(10, 5)       = {my_math.add(10, 5)}")
print(f"my_math.square(7)        = {my_math.square(7)}")
print(f"my_math.multiply(3, 4)   = {my_math.multiply(3, 4)}")
print()
print("📌 Зверніть увагу: блок 'if __name__ == __main__' НЕ виконався")
print("   бо ми зробили import, а не python my_math.py")

In [ ]:
# Приклад 2.3: Явна перевірка __name__ при розробці

def run_demo(module_name: str, is_main: bool):
    """Симулює поведінку модуля залежно від контексту."""
    print(f"--- {module_name}: __name__ = {'__main__' if is_main else module_name!r} ---")
    if is_main:
        print("  ▶ Прямий запуск: виконується блок if __name__ == '__main__':")
    else:
        print("  ▶ Імпорт: блок if __name__ == '__main__': ПРОПУСКАЄТЬСЯ")
    print()

print("=== Порівняння контекстів ===")
print()
run_demo("calculator", is_main=True)   # як python calculator.py
run_demo("calculator", is_main=False)  # як import calculator
run_demo("utils", is_main=True)
run_demo("utils", is_main=False)

print("✅ Правило: ЗАВЖДИ захищайте точку входу через if __name__ == '__main__':")

---

## 🔍 Частина 3: Де Python шукає модулі (`sys.path`)

### Порядок пошуку

Коли ви пишете `import random`, Python перевіряє місця у **строгому порядку**:

```
┌─────────────────────────────────────────────────────────────────┐
│  import random   — де шукати?                                   │
│                                                                 │
│  1. Поточний каталог (де знаходиться ваш скрипт)                │
│  2. PYTHONPATH (змінна середовища, якщо встановлена)            │
│  3. Стандартні бібліотеки Python (stdlib)                       │
│  4. Встановлені пакети (site-packages)                          │
│                                                                 │
│  ▶ Перший знайдений файл перемагає — решта ігнорується!         │
└─────────────────────────────────────────────────────────────────┘
```

### ⚠️ Import Shadowing — небезпека тіньового імпорту

Якщо у вашому проекті є файл з іменем стандартного модуля — він **перекриє** стандартну бібліотеку:

```
my_project/
    random.py      ← ваш файл (без randint!)
    main.py        ← import random → завантажується ВАШ random.py!
```

| Версія Python | Поведінка |
|---------------|-----------|
| < 3.10        | Локальний `random.py` перекриває stdlib → `AttributeError` |
| 3.10+         | stdlib захищена — `random` завантажиться правильно |
| Будь-яка      | Власні модулі (`myutils.py`) **не захищені** в жодній версії |

### ❌ Небезпечні імена для файлів

```
random.py   math.py   sys.py   os.py   string.py
time.py     json.py   csv.py   io.py   re.py
```

**Правило:** Не називайте свої файли іменами стандартних модулів Python.

In [ ]:
# Приклад 3.1: Перегляд sys.path
import sys

print("=== sys.path — місця пошуку модулів ===")
print()
for i, path in enumerate(sys.path):
    label = ""
    if i == 0:
        label = "  ← поточний каталог (найвищий пріоритет!)"
    elif "site-packages" in path:
        label = "  ← встановлені пакети (pip install)"
    elif "lib" in path.lower():
        label = "  ← стандартна бібліотека"
    print(f"  [{i}] {path}{label}")
print()
print(f"Всього шляхів: {len(sys.path)}")

In [ ]:
# Приклад 3.2: Знайти файл завантаженого модуля
import math
import os
import sys

print("=== Де знаходяться завантажені модулі? ===")
print()

modules_to_check = ["math", "os", "sys", "json"]
for mod_name in modules_to_check:
    mod = sys.modules.get(mod_name)
    if mod and hasattr(mod, "__file__") and mod.__file__:
        print(f"  {mod_name:<8} → {mod.__file__}")
    elif mod:
        print(f"  {mod_name:<8} → (вбудований модуль, без файлу)")
print()
print("💡 Підказка: якщо підозрюєте тіньовий імпорт — перевірте __file__:")
print("   import random")
print("   print(random.__file__)   # покаже який файл насправді завантажено")

In [ ]:
# Приклад 3.3: Перевірка та очищення sys.modules
import sys

print("=== Кеш sys.modules ===")
print()

# Скільки модулів зараз завантажено
print(f"Завантажено модулів: {len(sys.modules)}")
print()

# Кілька модулів, що точно є
for name in ["math", "os", "sys", "json", "collections"]:
    in_cache = name in sys.modules
    print(f"  '{name}': {'✅ в кеші' if in_cache else '❌ не завантажений'}")
print()
print("💡 Щоб примусово перезавантажити модуль (при розробці):")
print("   import importlib")
print("   importlib.reload(my_module)")
print()
print("💡 Щоб видалити з кешу (для тестів):")
print("   del sys.modules['my_module']")
print("   import my_module  # тепер завантажиться заново")

---

## ⌨️ Частина 4: Аргументи командного рядка (`sys.argv`)

### Що таке `sys.argv`?

Список `sys.argv` містить **аргументи, передані програмі при запуску**:

```bash
python script.py Alice 42 --verbose
#        ↑           ↑   ↑   ↑
#  sys.argv[0]     [1] [2] [3]
```

| Індекс | Значення | Тип |
|--------|----------|-----|
| `sys.argv[0]` | Ім'я скрипта | `str` |
| `sys.argv[1]` | Перший аргумент | `str` |
| `sys.argv[2]` | Другий аргумент | `str` |
| `sys.argv[n]` | n-й аргумент | `str` |

### ⚠️ Ключовий момент: усі аргументи — рядки!

```python
# Запуск: python calc.py 10 20
print(sys.argv[1] + sys.argv[2])   # → '1020'  (конкатенація рядків!)
print(int(sys.argv[1]) + int(sys.argv[2]))  # → 30  (правильно!)
```

### Безпечна обробка аргументів

```python
import sys

if __name__ == "__main__":
    # 1. Перевіряємо кількість аргументів
    if len(sys.argv) != 3:
        print(f"Використання: python {sys.argv[0]} <ім'я> <вік>")
        sys.exit(1)           # завершення з кодом помилки

    name = sys.argv[1]        # рядок — без конвертації
    age_str = sys.argv[2]

    # 2. Перевіряємо та конвертуємо типи
    if not age_str.isdigit():
        print("Помилка: вік має бути цілим числом")
        sys.exit(1)

    age = int(age_str)
    print(f"{name}, тобі {age} років")
```

In [ ]:
# Приклад 4.1: Симуляція sys.argv
import sys

print("=== sys.argv у Jupyter ===")
print(f"sys.argv = {sys.argv}")
print(f"sys.argv[0] = {sys.argv[0]!r}  (ім'я скрипта/ядра)")
print()

# Симулюємо запуск: python script.py Alice 42
print("Симуляція: python script.py Alice 42")
fake_argv = ["script.py", "Alice", "42"]

print(f"  sys.argv[0] = {fake_argv[0]!r}")
print(f"  sys.argv[1] = {fake_argv[1]!r}  (тип: {type(fake_argv[1]).__name__})")
print(f"  sys.argv[2] = {fake_argv[2]!r}  (тип: {type(fake_argv[2]).__name__})")
print(f"  len(sys.argv) = {len(fake_argv)}")

In [ ]:
# Приклад 4.2: Пастка з типами — рядки vs числа
print("=== Пастка: sys.argv завжди повертає рядки ===")
print()

# Симулюємо sys.argv[1] = "10", sys.argv[2] = "20"
a_str, b_str = "10", "20"

print("Якщо НЕ конвертувати:")
result_wrong = a_str + b_str
print(f"  '10' + '20' = {result_wrong!r}  ← КОНКАТЕНАЦІЯ рядків!")
print()

print("Якщо конвертувати через int():")
result_correct = int(a_str) + int(b_str)
print(f"  int('10') + int('20') = {result_correct}  ← правильно")
print()

print("Типи різних конвертацій:")
print(f"  int('42')   = {int('42')!r}     тип: {type(int('42')).__name__}")
print(f"  float('3.14') = {float('3.14')!r}  тип: {type(float('3.14')).__name__}")
print(f"  '42'        = {'42'!r}      тип: {type('42').__name__}")

In [ ]:
# Приклад 4.3: Безпечний парсер аргументів
import sys

def parse_two_ints(argv):
    """
    Парсить два цілі числа з argv[1] та argv[2].
    Повертає (ok: bool, result) де result — (a, b) або повідомлення помилки.
    """
    if len(argv) != 3:
        return False, f"Очікується 2 аргументи, отримано {len(argv) - 1}"

    a_str, b_str = argv[1], argv[2]

    if not a_str.lstrip('-').isdigit():
        return False, f"Перший аргумент '{a_str}' не є цілим числом"

    if not b_str.lstrip('-').isdigit():
        return False, f"Другий аргумент '{b_str}' не є цілим числом"

    return True, (int(a_str), int(b_str))


# Тестуємо різні сценарії
test_cases = [
    ["script.py", "10", "20"],       # ✅ нормальний випадок
    ["script.py", "10"],             # ❌ мало аргументів
    ["script.py", "abc", "20"],      # ❌ не число
    ["script.py", "-5", "100"],      # ✅ від'ємне число
    ["script.py", "3.14", "2"],      # ❌ дробове
]

print("=== Тести безпечного парсера ===")
for argv in test_cases:
    ok, result = parse_two_ints(argv)
    args_str = " ".join(argv[1:])
    if ok:
        a, b = result
        print(f"  python script.py {args_str:<12} → ✅ a={a}, b={b}, сума={a+b}")
    else:
        print(f"  python script.py {args_str:<12} → ❌ {result}")

---

## 💻 Частина 5: Консольні програми (CLI) з `while True:`

### Типова структура CLI-програми

```python
import sys

def main():
    print("Програма запущена. Введіть 'exit' для виходу.")

    while True:                              # нескінченний цикл
        raw = input("Введіть число: ")       # отримуємо ввід
        cleaned = raw.strip()                # видаляємо пробіли

        if cleaned.lower() == "exit":        # вихід
            print("До побачення!")
            break

        if not cleaned.isdigit():            # валідація
            print("⚠️  Введіть ціле позитивне число")
            continue                         # пропускаємо — нова ітерація

        number = int(cleaned)                # конвертація
        print(f"Результат: {number} × 2 = {number * 2}")

if __name__ == "__main__":
    main()
```

### Методи рядка для валідації (без `try/except`)

| Метод | Повертає `True` якщо... | Приклади |
|-------|--------------------------|----------|
| `.isdigit()` | тільки цифри `0-9` | `"42"` ✅, `"-5"` ❌, `"3.14"` ❌ |
| `.isalpha()` | тільки літери | `"Alice"` ✅, `"Al 1"` ❌ |
| `.isalnum()` | цифри і літери | `"ABC123"` ✅, `"abc!"` ❌ |
| `.strip()` | (видаляє пробіли по краях) | `"  10  "` → `"10"` |
| `.lower()` | (всі символи малими) | `"EXIT"` → `"exit"` |
| `.lstrip('-').isdigit()` | ціле число (включно з від'ємним) | `"-5"` ✅ |

### Ключові оператори циклу

| Оператор | Дія |
|----------|-----|
| `break` | Виходить з циклу повністю |
| `continue` | Пропускає поточну ітерацію, переходить до наступної |
| `sys.exit(0)` | Завершує програму (код 0 = успіх, 1 = помилка) |

In [ ]:
# Приклад 5.1: Методи рядка для валідації
print("=== Методи рядка для валідації ===")
print()

test_values = ["42", "  42  ", "ten", "-5", "3.14", "0", "", "ABC123", "Alice"]

print(f"{'Рядок':<12} {'strip()':>10} {'isdigit()':>10} {'isalpha()':>10} {'isalnum()':>10}")
print("-" * 60)
for s in test_values:
    stripped = s.strip()
    print(f"{s!r:<12} {stripped!r:>10} {str(stripped.isdigit()):>10} "
          f"{str(stripped.isalpha()):>10} {str(stripped.isalnum()):>10}")

In [ ]:
# Приклад 5.2: Симуляція CLI-циклу з різними вводами
print("=== Симуляція CLI-калькулятора ===")
print()

# Симульовані вводи користувача
fake_inputs = ["  ", "abc", "-5", "0", "  42  ", "100", "exit"]

for raw in fake_inputs:
    print(f"Користувач вводить: {raw!r}")
    cleaned = raw.strip()

    if cleaned.lower() == "exit":
        print("  → Вихід з програми")
        break

    if not cleaned:
        print("  → ⚠️  Порожній ввід, повторіть")
        continue

    if not cleaned.isdigit():
        print(f"  → ⚠️  '{cleaned}' не є цілим позитивним числом")
        continue

    number = int(cleaned)
    if number == 0:
        print(f"  → ⚠️  Число має бути більше 0")
        continue

    print(f"  → ✅ {number} × 2 = {number * 2}")

In [ ]:
# Приклад 5.3: Повний CLI-калькулятор (4 операції)
print("=== Симуляція: CLI-калькулятор з 4 операціями ===")
print()

def calculate(a: float, op: str, b: float):
    """Виконує операцію. Повертає (ok, result)."""
    if op == "+":
        return True, a + b
    elif op == "-":
        return True, a - b
    elif op == "*":
        return True, a * b
    elif op == "/":
        if b == 0:
            return False, "Ділення на нуль!"
        return True, a / b
    else:
        return False, f"Невідома операція '{op}'"

def parse_line(line: str):
    """Парсить рядок '10 + 5' → (10.0, '+', 5.0) або None."""
    parts = line.strip().split()
    if len(parts) != 3:
        return None
    try:
        a = float(parts[0])
        b = float(parts[2])
        return a, parts[1], b
    except ValueError:
        return None

# Симульовані вводи
fake_session = [
    "10 + 5",
    "100 / 4",
    "7 * 8",
    "10 / 0",
    "abc + 5",
    "exit",
]

for line in fake_session:
    print(f"  ввід: {line!r}")
    if line.strip().lower() == "exit":
        print("  → До побачення!")
        break
    parsed = parse_line(line)
    if parsed is None:
        print("  → ⚠️  Формат: <число> <операція> <число>")
        continue
    a, op, b = parsed
    ok, result = calculate(a, op, b)
    if ok:
        print(f"  → ✅ {a} {op} {b} = {result}")
    else:
        print(f"  → ❌ {result}")

In [ ]:
# Приклад 5.4: CLI з меню (menu-driven program)
print("=== Симуляція: CLI з меню ===")
print()

MENU = """
╔══════════════════════════════╗
║   🔢 Числовий аналізатор     ║
╠══════════════════════════════╣
║  1. Перевірити парність      ║
║  2. Знайти множники          ║
║  3. Перевірити простоту      ║
║  0. Вихід                    ║
╚══════════════════════════════╝"""

def is_even(n): return n % 2 == 0
def factors(n): return [i for i in range(1, n + 1) if n % i == 0]
def is_prime(n):
    if n < 2: return False
    return all(n % i != 0 for i in range(2, int(n**0.5) + 1))

# Симулюємо сесію меню
fake_session = [
    ("1", "12"),   # парність 12
    ("2", "12"),   # множники 12
    ("3", "17"),   # простота 17
    ("3", "12"),   # простота 12
    ("0", ""),     # вихід
]

print(MENU)

for choice, num_str in fake_session:
    print(f"\n  Вибір: {choice!r}")

    if choice == "0":
        print("  → До побачення!")
        break
    elif choice not in ("1", "2", "3"):
        print("  → ⚠️  Виберіть 0-3")
        continue

    print(f"  Число: {num_str!r}")
    if not num_str.isdigit():
        print("  → ⚠️  Введіть ціле число")
        continue

    n = int(num_str)
    if choice == "1":
        print(f"  → {n} {'парне' if is_even(n) else 'непарне'}")
    elif choice == "2":
        print(f"  → Множники {n}: {factors(n)}")
    elif choice == "3":
        print(f"  → {n} {'просте' if is_prime(n) else 'не просте'} число")

---

## 📋 Шпаргалка (Quick Reference)

### 1. Форми імпорту
```python
import math                         # весь модуль
from math import sqrt, pi           # конкретні символи
import numpy as np                  # псевдонім
from math import sqrt as square_root  # псевдонім для функції
```

### 2. `__name__` guard — захист точки входу
```python
# my_module.py
def my_function(): ...

if __name__ == "__main__":   # виконується ТІЛЬКИ при python my_module.py
    my_function()            # при import — цей блок ігнорується
```

### 3. sys.argv — аргументи командного рядка
```python
import sys

# python script.py Alice 25
name  = sys.argv[1]          # 'Alice'   (завжди str!)
age   = int(sys.argv[2])     # 25        (конвертуємо явно)
count = len(sys.argv)        # 3         (ім'я скрипта + 2 аргументи)

# Безпечний доступ:
if len(sys.argv) < 3:
    print("Потрібно 2 аргументи")
    sys.exit(1)
```

### 4. Структура CLI-програми
```python
import sys

def main():
    while True:
        raw = input("Введіть: ").strip()

        if raw.lower() in ("exit", "quit", "q"):
            break

        if not raw.isdigit():        # валідація
            print("Помилка")
            continue

        process(int(raw))            # обробка

if __name__ == "__main__":
    main()
```

### 5. Валідація рядків
```python
"42".isdigit()           # True   — невід'ємне ціле
"-5".isdigit()           # False  ⚠️
"-5".lstrip('-').isdigit()  # True — ціле (включно з від'ємним)
"abc".isalpha()          # True   — тільки літери
"abc123".isalnum()       # True   — літери + цифри
"  42  ".strip()         # '42'   — без пробілів по краях
```

### 6. Корисні інструменти налагодження
```python
import sys, importlib

print(sys.path)              # де Python шукає модулі
print(module.__file__)       # де знаходиться файл модуля
print(sys.modules.keys())    # список завантажених модулів
importlib.reload(module)     # примусово перезавантажити модуль
```

---

# 📝 Задачі — Самостійна робота

### TODO 1: Написати власний модуль

Доповніть `my_math.py` функцією `average(numbers)`, яка приймає список чисел і повертає середнє значення.  
Захистіть тест через `if __name__ == "__main__":`.  
(Відредагуйте файл `my_math.py` у цій папці)

In [ ]:
# Після додавання average() до my_math.py перезапустіть цю клітинку
import importlib
import my_math
importlib.reload(my_math)

# YOUR CODE: перевірте що average() існує і працює
# print(my_math.average([1, 2, 3, 4, 5]))  # очікується 3.0

### TODO 2: Виправте CLI-код

У коді нижче є помилки. Знайдіть та виправте їх.

In [ ]:
# ⚠️ Цей код містить помилки — знайдіть і виправте
import sys

def process_args():
    # Помилка 1: немає перевірки len(sys.argv)
    name = sys.argv[1]          # ← падає якщо аргумент не переданий

    # Помилка 2: аргументи не є числами автоматично
    score = sys.argv[2]         # ← потребує int()
    doubled = score * 2         # ← рядок × 2 = "1010", не 20!

    print(f"{name}: {doubled}")

# Правильна версія — напишіть нижче:
# YOUR CODE HERE

### TODO 3: Допишіть валідацію

Функція `ask_positive_int()` повинна запитувати у користувача число доки він не введе **ціле позитивне** число (> 0). Без `try/except`.

In [ ]:
# Симуляція кількох вводів для тестування
FAKE_INPUTS = iter(["abc", "-5", "0", "  42  "])  # симульовані відповіді

def ask_positive_int(prompt="Введіть число: "):
    """Повертає ціле позитивне число. Повторює запит при помилці."""
    while True:
        raw = next(FAKE_INPUTS, "1")  # у реальному коді: raw = input(prompt)
        # YOUR CODE HERE: валідація без try/except
        # Підказка: .strip(), .isdigit(), int() > 0
        pass  # видаліть цей рядок

# Тест (розкоментуйте після реалізації):
# result = ask_positive_int()
# print("Введене число:", result)  # очікується 42